# Two-Stage Separate U-Net — 5-Fold Cross-Validation Benchmark
> **Thesis:** Localization, Segmentation & Severity Estimation of Pumpkin Leaf Diseases  
> **Model Architecture:** Two-Stage Pipeline (Stage 1: Leaf U-Net · Stage 2: Lesion U-Net)  
> **Source Precedent:** Yao et al. [16] (Plum leaf extraction + lesion U-Net) & Zhao et al. [15]  
> **Evaluation Protocol:** 5-Fold Stratified Cross-Validation on 100 Pilot Images (20 images/fold: 4 per class)  
> **Objective:** Direct ablation against single-stage DualTaskUNet to test whether dual-head multi-task feature sharing outperforms two independent networks.


## Cell 0 — Setup Google Colab Environment & Dependencies


In [1]:
# Install segmentation-models-pytorch and openpyxl
!pip install -q segmentation-models-pytorch openpyxl

import os, sys, random, time
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
import segmentation_models_pytorch as smp

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PILOT_ZIP  = '/content/drive/MyDrive/pilot_100_images.zip'
PILOT_ROOT = '/content/pilot_100_images'

if not os.path.exists(PILOT_ROOT):
    print("Extracting pilot dataset...")
    !unzip -q "$PILOT_ZIP" -d /content/
    print("Extraction complete.")
else:
    print("Pilot folder already present.")

MASK_ROOT  = os.path.join(PILOT_ROOT, 'automated_masks')
LEAF_DIR   = os.path.join(MASK_ROOT, 'leaf')
LESION_DIR = os.path.join(MASK_ROOT, 'lesion')
OUT_DIR    = '/content/drive/MyDrive/pumpkin_severity_runs/twostage_unet_cv5'
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device} | Output Directory: {OUT_DIR}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.2 MB/s eta 0:00:00
Mounted at /content/drive
Extracting pilot dataset...
Extraction complete.
Device: cuda | Output Directory: /content/drive/MyDrive/pumpkin_severity_runs/twostage_unet_cv5


## Cell 1 — Stratified 5-Fold Dataset Manifest Creation


In [2]:
classes_5 = ['BL', 'DM', 'HL', 'MD', 'PM']
class_full = {
    'BL': 'Bacterial Leaf Spot',
    'DM': 'Downy Mildew',
    'HL': 'Healthy Leaf',
    'MD': 'Mosaic Disease',
    'PM': 'Powdery Mildew'
}

records = []
for fname in sorted(os.listdir(PILOT_ROOT)):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')): continue
    prefix = fname.split('_')[0]
    if prefix not in classes_5: continue

    stem = os.path.splitext(fname)[0]
    leaf_p = os.path.join(LEAF_DIR, f"{stem}_leaf.png")
    lesion_p = os.path.join(LESION_DIR, f"{stem}_lesion.png")

    if os.path.exists(leaf_p) and os.path.exists(lesion_p):
        records.append({
            'filename': fname,
            'stem': stem,
            'class_code': prefix,
            'class_name': class_full[prefix],
            'img_path': os.path.join(PILOT_ROOT, fname),
            'leaf_path': leaf_p,
            'lesion_path': lesion_p,
            'is_healthy': (prefix == 'HL')
        })

df_all = pd.DataFrame(records)
print(f"Total matched pilot samples: {len(df_all)}")

# Stratified 5-Fold Split (Seed 42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
df_all['fold'] = -1
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_all, df_all['class_code'])):
    df_all.loc[val_idx, 'fold'] = fold_idx

df_all.to_csv(os.path.join(OUT_DIR, 'cv5_splits.csv'), index=False)
print("5-Fold cross-validation assignment:")
print(pd.crosstab(df_all['fold'], df_all['class_code']))


Total matched pilot samples: 80
5-Fold cross-validation assignment:
class_code  DM  HL  MD  PM
fold                      
0            4   4   4   4
1            4   4   4   4
2            4   4   4   4
3            4   4   4   4
4            4   4   4   4


## Cell 2 — Dataset & Two-Stage Loader Definition


In [3]:
IMG_SIZE = (256, 256)

class PumpkinTwoStageDataset(Dataset):
    def __init__(self, df, img_size=(256, 256), is_train=True):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['img_path']).convert('RGB').resize(self.img_size, Image.BILINEAR)
        leaf = Image.open(row['leaf_path']).convert('L').resize(self.img_size, Image.NEAREST)
        lesion = Image.open(row['lesion_path']).convert('L').resize(self.img_size, Image.NEAREST)

        if self.is_train and random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            leaf = leaf.transpose(Image.FLIP_LEFT_RIGHT)
            lesion = lesion.transpose(Image.FLIP_LEFT_RIGHT)

        img_arr = np.array(img, dtype=np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img_t = torch.tensor((img_arr - mean) / std).permute(2, 0, 1).float()

        leaf_t   = (torch.tensor(np.array(leaf) > 127).float()).unsqueeze(0)
        lesion_t = (torch.tensor(np.array(lesion) > 127).float()).unsqueeze(0)
        lesion_t = lesion_t * leaf_t  # ground truth constraint

        return {
            'img': img_t,
            'leaf': leaf_t,
            'lesion': lesion_t,
            'is_healthy': row['is_healthy'],
            'filename': row['filename']
        }

print("Two-stage dataset class defined.")


Two-stage dataset class defined.


## Cell 3 — Two-Stage Model Architecture (Leaf U-Net & Lesion U-Net)


In [4]:
# We instantiate two separate, dedicated U-Net models
# Stage 1: Leaf U-Net (ResNet-34 encoder)
# Stage 2: Lesion U-Net (ResNet-34 encoder)
def build_leaf_model():
    return smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=1)

def build_lesion_model():
    return smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=1)

print("Stage 1 Leaf U-Net and Stage 2 Lesion U-Net architectures ready.")


Stage 1 Leaf U-Net and Stage 2 Lesion U-Net architectures ready.


## Cell 4 — Losses and Evaluation Metric Helpers


In [5]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2.0 * (probs * targets).sum(dim=(2, 3)) + self.smooth
        den = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + self.smooth
        return 1.0 - (num / den).mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.8, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = targets * probs + (1 - targets) * (1 - probs)
        return (self.alpha * ((1.0 - pt) ** self.gamma) * bce).mean()

def compute_metrics(p_bin, t_bin):
    inter = (p_bin & t_bin).sum().item()
    union = (p_bin | t_bin).sum().item()
    tot   = p_bin.sum().item() + t_bin.sum().item()
    if tot == 0: return 1.0, 1.0
    iou = inter / union if union > 0 else 0.0
    dice = 2.0 * inter / tot if tot > 0 else 0.0
    return iou, dice

print("Losses and evaluation functions ready.")


Losses and evaluation functions ready.


## Cell 5 — 5-Fold Cross-Validation Training Loop


In [6]:
EPOCHS = 20
BATCH_SIZE = 8
LR = 3e-4

bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()
focal_loss = FocalLoss()

fold_records = []

for fold in range(5):
    print(f"\n───────────────────────────────────────────────────────")
    print(f"FOLD {fold + 1} / 5 — Training Two-Stage Separate U-Net")
    print(f"───────────────────────────────────────────────────────")

    train_df = df_all[df_all['fold'] != fold]
    val_df   = df_all[df_all['fold'] == fold]

    train_loader = DataLoader(PumpkinTwoStageDataset(train_df, is_train=True), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(PumpkinTwoStageDataset(val_df, is_train=False), batch_size=BATCH_SIZE, shuffle=False)

    model_leaf = build_leaf_model().to(device)
    model_les  = build_lesion_model().to(device)

    opt_leaf = torch.optim.AdamW(model_leaf.parameters(), lr=LR, weight_decay=1e-4)
    opt_les  = torch.optim.AdamW(model_les.parameters(), lr=LR, weight_decay=1e-4)

    for ep in range(EPOCHS):
        model_leaf.train(); model_les.train()
        for b in train_loader:
            imgs = b['img'].to(device)
            leaf_gt = b['leaf'].to(device)
            les_gt  = b['lesion'].to(device)

            # Stage 1: Leaf U-Net optimization
            l_pred = model_leaf(imgs)
            l_loss = bce_loss(l_pred, leaf_gt) + dice_loss(l_pred, leaf_gt)
            opt_leaf.zero_grad(); l_loss.backward(); opt_leaf.step()

            # Stage 2: Lesion U-Net optimization
            les_pred = model_les(imgs)
            les_loss = focal_loss(les_pred, les_gt) + dice_loss(les_pred, les_gt)
            opt_les.zero_grad(); les_loss.backward(); opt_les.step()

    # Validation Evaluation on out-of-fold set (n=20)
    model_leaf.eval(); model_les.eval()
    leaf_dices, leaf_ious = [], []
    les_dices_all, les_ious_all = [], []
    les_dices_dis = []
    sev_errs_all, sev_errs_dis = [], []

    with torch.no_grad():
        for b in val_loader:
            imgs = b['img'].to(device)
            l_pred = model_leaf(imgs)
            les_pred = model_les(imgs)

            l_bin = (torch.sigmoid(l_pred) > 0.5).cpu().numpy()
            les_bin = (torch.sigmoid(les_pred) > 0.5).cpu().numpy()

            for i in range(len(b['img'])):
                cur_l_p = l_bin[i, 0]
                # Two-stage pipeline constraint: lesion prediction masked by predicted leaf
                cur_les_p = les_bin[i, 0] & cur_l_p
                cur_l_t = (b['leaf'][i, 0].numpy() > 0.5)
                cur_les_t = (b['lesion'][i, 0].numpy() > 0.5)
                is_hl = b['is_healthy'][i]

                iou_l, dice_l = compute_metrics(cur_l_p, cur_l_t)
                iou_les, dice_les = compute_metrics(cur_les_p, cur_les_t)

                leaf_dices.append(dice_l); leaf_ious.append(iou_l)
                les_dices_all.append(dice_les); les_ious_all.append(iou_les)

                gt_sev = (cur_les_t.sum() / max(1, cur_l_t.sum())) * 100.0
                pr_sev = (cur_les_p.sum() / max(1, cur_l_p.sum())) * 100.0
                err = abs(pr_sev - gt_sev)
                sev_errs_all.append(err)

                if not is_hl:
                    les_dices_dis.append(dice_les)
                    sev_errs_dis.append(err)

    fold_rec = {
        'fold': fold + 1,
        'leaf_dice': np.mean(leaf_dices),
        'leaf_iou': np.mean(leaf_ious),
        'lesion_dice_all': np.mean(les_dices_all),
        'lesion_dice_dis': np.mean(les_dices_dis),
        'sev_mae_all': np.mean(sev_errs_all),
        'sev_mae_dis': np.mean(sev_errs_dis)
    }
    fold_records.append(fold_rec)
    print(f"Fold {fold+1} Results: Leaf Dice={fold_rec['leaf_dice']:.4f} | Lesion Dice(Dis)={fold_rec['lesion_dice_dis']:.4f} | Sev MAE(Dis)={fold_rec['sev_mae_dis']:.2f}% | Sev MAE(All)={fold_rec['sev_mae_all']:.2f}%")

df_folds = pd.DataFrame(fold_records)
df_folds.to_csv(os.path.join(OUT_DIR, 'twostage_unet_fold_metrics.csv'), index=False)



───────────────────────────────────────────────────────
FOLD 1 / 5 — Training Two-Stage Separate U-Net
───────────────────────────────────────────────────────


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Fold 1 Results: Leaf Dice=0.9554 | Lesion Dice(Dis)=0.8028 | Sev MAE(Dis)=1.69% | Sev MAE(All)=1.55%

───────────────────────────────────────────────────────
FOLD 2 / 5 — Training Two-Stage Separate U-Net
───────────────────────────────────────────────────────
Fold 2 Results: Leaf Dice=0.9557 | Lesion Dice(Dis)=0.7921 | Sev MAE(Dis)=3.56% | Sev MAE(All)=3.49%

───────────────────────────────────────────────────────
FOLD 3 / 5 — Training Two-Stage Separate U-Net
───────────────────────────────────────────────────────
Fold 3 Results: Leaf Dice=0.9525 | Lesion Dice(Dis)=0.7957 | Sev MAE(Dis)=3.26% | Sev MAE(All)=2.65%

───────────────────────────────────────────────────────
FOLD 4 / 5 — Training Two-Stage Separate U-Net
───────────────────────────────────────────────────────
Fold 4 Results: Leaf Dice=0.9497 | Lesion Dice(Dis)=0.8102 | Sev MAE(Dis)=3.47% | Sev MAE(All)=4.96%

───────────────────────────────────────────────────────
FOLD 5 / 5 — Training Two-Stage Separate U-Net
────────────

## Cell 6 — Aggregate 5-Fold Summary (Mean ± Standard Deviation)


In [7]:
print("============================================================")
print("  TWO-STAGE SEPARATE U-NET — 5-FOLD CV SUMMARY")
print("============================================================")
print(f"Leaf Dice (Mean ± Std)          : {df_folds['leaf_dice'].mean():.4f} ± {df_folds['leaf_dice'].std():.4f}")
print(f"Leaf IoU  (Mean ± Std)          : {df_folds['leaf_iou'].mean():.4f} ± {df_folds['leaf_iou'].std():.4f}")
print(f"Lesion Dice [Diseased] (Mean±Std): {df_folds['lesion_dice_dis'].mean():.4f} ± {df_folds['lesion_dice_dis'].std():.4f}")
print(f"Lesion Dice [Overall]  (Mean±Std): {df_folds['lesion_dice_all'].mean():.4f} ± {df_folds['lesion_dice_all'].std():.4f}")
print(f"Severity MAE [Diseased] (Mean±Std): {df_folds['sev_mae_dis'].mean():.2f}% ± {df_folds['sev_mae_dis'].std():.2f}%")
print(f"Severity MAE [Overall]  (Mean±Std): {df_folds['sev_mae_all'].mean():.2f}% ± {df_folds['sev_mae_all'].std():.2f}%")
print("============================================================")

summary_dict = {
    'Model': 'Two-Stage Separate U-Net (Yao et al. [16])',
    'Leaf_Dice': f"{df_folds['leaf_dice'].mean():.4f} ± {df_folds['leaf_dice'].std():.4f}",
    'Leaf_IoU': f"{df_folds['leaf_iou'].mean():.4f} ± {df_folds['leaf_iou'].std():.4f}",
    'Lesion_Dice_Dis': f"{df_folds['lesion_dice_dis'].mean():.4f} ± {df_folds['lesion_dice_dis'].std():.4f}",
    'Lesion_Dice_All': f"{df_folds['lesion_dice_all'].mean():.4f} ± {df_folds['lesion_dice_all'].std():.4f}",
    'Severity_MAE_Dis': f"{df_folds['sev_mae_dis'].mean():.2f}% ± {df_folds['sev_mae_dis'].std():.2f}%",
    'Severity_MAE_All': f"{df_folds['sev_mae_all'].mean():.2f}% ± {df_folds['sev_mae_all'].std():.2f}%"
}
pd.DataFrame([summary_dict]).to_csv(os.path.join(OUT_DIR, 'summary_mean_std.csv'), index=False)


  TWO-STAGE SEPARATE U-NET — 5-FOLD CV SUMMARY
Leaf Dice (Mean ± Std)          : 0.9540 ± 0.0028
Leaf IoU  (Mean ± Std)          : 0.9133 ± 0.0045
Lesion Dice [Diseased] (Mean±Std): 0.8019 ± 0.0079
Lesion Dice [Overall]  (Mean±Std): 0.6014 ± 0.0059
Severity MAE [Diseased] (Mean±Std): 3.21% ± 0.91%
Severity MAE [Overall]  (Mean±Std): 3.72% ± 1.77%
